# Energy AI Hackathon 2026 — Solship
## Step-by-step walkthrough: Load → Clean → EDA → Forecast → MPC

## CELL 1 — Imports & constants

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import holidays
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ── System constants (from brief) ─────────────────────────────────────────────
DELTA_T               = 0.25        # h per timestep
BATTERY_CAPACITY_KWH  = 16.0
BATTERY_MAX_POWER_KW  = 8.0
GRID_LIMIT_KW         = 6.0
BATTERY_EFFICIENCY    = np.sqrt(0.90)   # ~0.9487 per direction
INITIAL_SOC           = 0.50

print('Libraries loaded ✓')
print(f'Battery efficiency per direction: {BATTERY_EFFICIENCY:.4f}')

## CELL 2 — Load the raw dataset

In [ ]:
# ── Kaggle path (update if running locally) ───────────────────────────────────
DATA_PATH = '/kaggle/input/energy-hackathon/ENERGY_Hackathon_DataSet(Sheet1).csv'

# ── Key quirks discovered during inspection ───────────────────────────────────
# 1. Separator is semicolon, not comma
# 2. Decimal separator is European comma  e.g. '3,04' → 3.04
# 3. Both 2024 and 2025 are in a single file
# 4. No buy_price column → must compute from Italian ToU tariff

raw = pd.read_csv(DATA_PATH, sep=';', decimal=',')
raw['timestamp'] = pd.to_datetime(raw['timestamp'])
raw = raw.sort_values('timestamp').reset_index(drop=True)

print('Shape:', raw.shape)
print('Columns:', list(raw.columns))
print()
raw.head(5)

## CELL 3 — Data quality audit

In [ ]:
print('=== YEAR DISTRIBUTION ===')
print(raw.groupby(raw['timestamp'].dt.year).size().rename('rows'))

print()
print('=== NULLS ===')
print(raw.isnull().sum())

print()
print('=== DUPLICATE TIMESTAMPS ===')
dups = raw[raw['timestamp'].duplicated(keep=False)]
print(f'{raw["timestamp"].duplicated().sum()} duplicate entries')
print(dups[['timestamp','battery_p','grid_p','load_p','pv_p','Selling_price_eur_kwh']].to_string())

print()
print('=== TIME GAPS (expected 15-min steps) ===')
diffs = raw['timestamp'].diff().dropna()
gaps = diffs[diffs != pd.Timedelta('15min')]
print(f'{len(gaps)} unexpected gaps:')
for idx, val in gaps.items():
    print(f'  row {idx}: {raw.loc[idx-1,"timestamp"]} → {raw.loc[idx,"timestamp"]}  ({val})')

print()
print('=== RANGE CHECK ===')
for col in ['load_p','pv_p','battery_p','grid_p','Selling_price_eur_kwh']:
    print(f'  {col:<28} min={raw[col].min():.3f}  max={raw[col].max():.3f}  nulls={raw[col].isnull().sum()}')

print()
print('=== GRID LIMIT VIOLATIONS (> 6 kW) ===')
print(f'  grid_p > 6:   {(raw["grid_p"] > GRID_LIMIT_KW).sum()} rows  (max={raw["grid_p"].max():.2f} kW)')
print(f'  grid_p < -6:  {(raw["grid_p"] < -GRID_LIMIT_KW).sum()} rows')

print()
print('=== ENERGY BALANCE (load = pv + grid + battery) ===')
residual = raw['load_p'] - raw['pv_p'] - raw['grid_p'] - raw['battery_p']
print(f'  Mean residual : {residual.mean():.5f} kW  (ideal = 0)')
print(f'  Max |residual|: {residual.abs().max():.3f} kW')
print(f'  Rows |res|>0.1: {(residual.abs()>0.1).sum()}')

## CELL 4 — Clean & fix all issues

Issues to fix:
1. **Rename columns** to match brief schema
2. **DST duplicates** on Oct 28 2024 — keep first occurrence (before clock rolled back)
3. **DST gaps** on Mar 31 2024 and Mar 30 2025 — 4 missing 15-min slots each, forward-fill
4. **8 null sell prices** — forward-fill from previous valid value
5. **Compute buy_price** from Italian ToU tariff (not in raw data)
6. **Split into 2024 (train) and 2025 (test)**

In [ ]:
import holidays as hol

# ── 1. Rename columns ─────────────────────────────────────────────────────────
df = raw.rename(columns={
    'battery_p':            'p_battery_kw',
    'grid_p':               'grid_kw',
    'load_p':               'load_kw',
    'pv_p':                 'pv_kw',
    'Selling_price_eur_kwh':'sell_price',
})

# ── 2. Drop duplicate timestamps (DST fall-back: keep first / CET reading) ────
before = len(df)
df = df.drop_duplicates(subset='timestamp', keep='first').reset_index(drop=True)
print(f'Dropped {before - len(df)} duplicate rows')

# ── 3. Reindex to complete 15-min grid, forward-fill DST gaps ─────────────────
full_index = pd.date_range(
    start=df['timestamp'].min(),
    end=df['timestamp'].max(),
    freq='15min'
)
df = df.set_index('timestamp').reindex(full_index).ffill().reset_index()
df = df.rename(columns={'index': 'timestamp'})
print(f'After reindex: {len(df)} rows  (expected ~{len(full_index)})')

# ── 4. Forward-fill remaining null sell prices ────────────────────────────────
df['sell_price'] = df['sell_price'].ffill()
print(f'Null sell prices remaining: {df["sell_price"].isnull().sum()}')

# ── 5. Compute buy_price from Italian ToU tariff ──────────────────────────────
italy_hols = hol.Italy(years=[2024, 2025])

def get_buy_price(ts):
    h = ts.hour
    d = ts.dayofweek   # 0=Mon … 6=Sun
    is_hol = ts.date() in italy_hols
    if is_hol or d == 6:                                     return 0.2440  # F3
    if d <= 4 and 8 <= h < 19:                               return 0.2540  # F1
    if (d <= 4 and (7 <= h < 8 or 19 <= h < 23)) or \
       (d == 5 and 7 <= h < 23):                             return 0.2682  # F2
    return 0.2440                                                            # F3

df['buy_price'] = df['timestamp'].map(get_buy_price)
print(f'Buy price computed. Unique values: {sorted(df["buy_price"].unique())}')

# ── 6. Split 2024 / 2025 ──────────────────────────────────────────────────────
train_df = df[df['timestamp'].dt.year == 2024].reset_index(drop=True)
test_df  = df[df['timestamp'].dt.year == 2025].reset_index(drop=True)

print(f'\nTrain (2024): {len(train_df)} rows  {train_df["timestamp"].min().date()} → {train_df["timestamp"].max().date()}')
print(f'Test  (2025): {len(test_df)} rows  {test_df["timestamp"].min().date()} → {test_df["timestamp"].max().date()}')
print()
train_df.head(3)